# SysSim — ARIA Tutorial (2026-05-22)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AISysSim/SysSim/blob/lexu/demo-notebook/demo/aria_tutorial.ipynb)

SysSim estimates the step time and peak memory of LLM training on hardware
you don't have, without running real computation. This notebook walks through
five demos:

1. **Models** — Qwen3-8B vs. Llama-3-8B (dense)
2. **Configs** — Batch / Seqlen / TP / PP sweeps
3. **GPU vendor** — AMD MI300X (roofline → trained predictor)
4. **Precision** — FP8 (roofline → trained predictor)
5. **Cost model** — modifying `estimate_runtime()`

**Before you run anything:** _Runtime → Change runtime type → T4 GPU_.
(If the install cell fails on T4, fall back to L4 or A100.)


In [ ]:
import torch
assert torch.cuda.is_available(), (
    "SysSim requires a GPU runtime. "
    "Runtime → Change runtime type → T4 GPU (or L4/A100), then re-run."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Install SysSim (~3-5 min on a fresh runtime)

In [ ]:
import os, subprocess

if not os.path.exists("SysSim") and not os.path.basename(os.getcwd()) == "SysSim":
    subprocess.run(
        'git config --global url."https://github.com/".insteadOf "git@github.com:"',
        shell=True, check=True,
    )
    subprocess.run(
        "git clone -b lexu/demo-notebook --recurse-submodules "
        "https://github.com/AISysSim/SysSim.git",
        shell=True, check=True,
    )

if os.path.basename(os.getcwd()) != "SysSim":
    %cd SysSim

# Install only what the demo actually needs. We deliberately skip two
# pyproject.toml deps that fail on Colab T4:
#   - megatron-bridge: pulls in mamba-ssm + transformer-engine[core_cu13]
#     which try to compile from source (no sm_75 wheel, no matching CUDA
#     toolkit). Only used by HFModel paths; the demo uses YAML configs.
#   - flashinfer-python: only used by SysSim's profiling code (not the
#     demo's predictor-training path) and often lacks an sm_75 wheel.
!pip install -q megatron-core xgboost 2>&1 | tail -5
# Install syssim itself from source without re-resolving deps
!pip install -q --no-deps -e . 2>&1 | tail -5
import syssim
print(f"SysSim imported from: {syssim.__file__}")

In [ ]:
import sys
sys.path.insert(0, ".")
from demo import helpers
print("Helpers loaded:", helpers.helpers_loaded())

## §1. Models — Qwen3-8B vs. Llama-3-8B

Same hardware (H100 DGX), same parallelism (TP=2, DP=4). The simulator
is architecture-aware — GQA group count, MLP ratio, RoPE settings all
flow through.

In [ ]:
from syssim.training.spec import load_model_yaml

QWEN3 = "examples/configs/models/qwen3-8b.yaml"
LLAMA = "demo/configs/models/llama3-8b.yaml"

for path in (QWEN3, LLAMA):
    cfg = load_model_yaml(path)
    print(f"{path}:")
    print(f"  layers={cfg.num_layers}  hidden={cfg.hidden_size}  "
          f"heads={cfg.num_attention_heads} (GQA groups={cfg.num_query_groups})  "
          f"ffn={cfg.ffn_hidden_size}  vocab={cfg.vocab_size}")

In [ ]:
import syssim
import pandas as pd

HW = "examples/configs/hardware/dgx_h100.yaml"
PAR = syssim.ParallelismConfig(tp=2, dp=4)
TR = syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16")

rows = []
for name, path in [("Qwen3-8B", QWEN3), ("Llama-3-8B", LLAMA)]:
    r = syssim.simulate(model=path, hardware=HW, parallelism=PAR, training=TR)
    rows.append({
        "model": name,
        "step_time_ms": round(r.step_time_ms, 2),
        "forward_ms": round(r.forward_ms, 2),
        "backward_ms": round(r.backward_ms, 2),
        "mfu": round(r.mfu, 3),
        "peak_memory_gb": round(r.peak_memory_gb, 2),
    })
pd.DataFrame(rows)

## §2. Configs — Batch / Seqlen / TP / PP

Hold model = Qwen3-8B and HW = H100 DGX fixed. Sweep one knob at a time.

In [ ]:
import matplotlib.pyplot as plt

def run_sweep(axis_label, axis_key, values):
    rows = []
    for v in values:
        par = syssim.ParallelismConfig(tp=2, dp=4)
        tr = syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16")
        if axis_key == "micro_batch":
            tr = syssim.TrainingConfig(micro_batch=v, global_batch=max(8, v*4), dtype="bf16")
        elif axis_key == "tp":
            par = syssim.ParallelismConfig(tp=v, dp=8 // v)
        elif axis_key == "pp":
            par = syssim.ParallelismConfig(pp=v, dp=8 // v)
        # NOTE: seq_length sweep below requires generating temp model YAMLs
        # because seq_length lives in the model YAML, not TrainingConfig.
        r = syssim.simulate(model=QWEN3, hardware=HW, parallelism=par, training=tr)
        rows.append({axis_label: v, "step_time_ms": round(r.step_time_ms, 2),
                     "peak_memory_gb": round(r.peak_memory_gb, 2), "mfu": round(r.mfu, 3)})
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar([str(v) for v in df[axis_label]], df["step_time_ms"])
    ax.set_xlabel(axis_label); ax.set_ylabel("step_time_ms")
    ax.set_title(f"Sweep: {axis_label}")
    plt.show()
    return df

for label, key, vals in [
    ("micro_batch", "micro_batch", [1, 2, 4]),
    ("TP",          "tp",          [1, 2, 4]),
    ("PP",          "pp",          [1, 2, 4]),
]:
    print(f"\n=== Sweep: {label} ===")
    display(run_sweep(label, key, vals))

In [ ]:
# Seqlen sweep: generate temp model YAMLs with different seq_length values
import tempfile, yaml
from pathlib import Path

with open(QWEN3) as _f:
    base_cfg = yaml.safe_load(_f)
seq_rows = []
with tempfile.TemporaryDirectory() as tmp:
    for seq in (2048, 4096, 8192):
        cfg = dict(base_cfg); cfg["seq_length"] = seq
        path = Path(tmp) / f"qwen3-8b_seq{seq}.yaml"
        path.write_text(yaml.dump(cfg))
        r = syssim.simulate(model=str(path), hardware=HW,
                            parallelism=syssim.ParallelismConfig(tp=2, dp=4),
                            training=syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="bf16"))
        seq_rows.append({"seq_length": seq, "step_time_ms": round(r.step_time_ms, 2),
                         "mfu": round(r.mfu, 3)})

seq_df = pd.DataFrame(seq_rows)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([str(v) for v in seq_df["seq_length"]], seq_df["step_time_ms"])
ax.set_xlabel("seq_length"); ax.set_ylabel("step_time_ms"); ax.set_title("Sweep: seq_length")
plt.show()
seq_df

## §3. GPU vendor — AMD MI300X (Mike)

Three sub-demos:
1. **Roofline-only** on `mi300x.yaml`
2. **Synthesize MI300X profiling CSVs** in-cell
3. **Train predictor → re-run**, compare Roofline vs. Trained

Because we're running on a Colab GPU that isn't MI300X, we use a
`simulated_hardware()` context manager (defined in `demo/helpers.py`)
to override SysSim's hardware auto-detection while training the predictor.
In production you'd profile on the actual MI300X and skip the override.

In [ ]:
# §3a: Roofline-only
MI300X = "demo/configs/hardware/mi300x.yaml"
PAR_MI = syssim.ParallelismConfig(tp=8)

print("=== MI300X — Roofline-only (no efficiency model) ===")
r_roof_mi = syssim.simulate(model=QWEN3, hardware=MI300X, parallelism=PAR_MI, training=TR)
print(f"step_time_ms={r_roof_mi.step_time_ms:.2f}  mfu={r_roof_mi.mfu:.3f}  "
      f"peak_mem_gb={r_roof_mi.peak_memory_gb:.2f}")
roofline_ms_mi = r_roof_mi.step_time_ms

In [ ]:
# §3b: Synthesize MI300X profiling data
import tempfile
from pathlib import Path

MI300X_PEAKS = {
    "peak_tflops_mm": 1307.0, "peak_tflops_math": 163.4,
    "peak_memory_bandwidth_gbps": 5300.0, "peak_tflops_mm_fp8": 2615.0,
    "peak_tflops_mm_fp4": None,
}

PROF_DIR_MI = Path(tempfile.mkdtemp(prefix="mi300x_profiling_"))
helpers.synthesize_gemm_csv(PROF_DIR_MI / "gemm_mi300x_fp16_data.csv",
                            peak_tflops=1307, peak_bw_GBps=5300, dtype_bytes=2)
helpers.synthesize_attn_csv(PROF_DIR_MI / "attn_mi300x_fp16_data.csv",
                            peak_tflops=1307, peak_bw_GBps=5300, dtype_bytes=2)
helpers.synthesize_rmsnorm_csv(PROF_DIR_MI / "rmsnorm_mi300x_fp16_data.csv",
                               peak_bw_GBps=5300, dtype_bytes=2)
print(f"Wrote profiling CSVs to {PROF_DIR_MI}:")
for p in sorted(PROF_DIR_MI.glob("*.csv")):
    print(f"  {p.name}  ({p.stat().st_size} bytes)")

In [ ]:
# §3c: Train predictor → swap → re-simulate
from syssim.compute.compute_cost_profiler import train_efficiency_model
from syssim.api import set_efficiency_model_dir

MODEL_DIR_MI = Path(tempfile.mkdtemp(prefix="mi300x_models_"))

try:
    with helpers.simulated_hardware("mi300x", MI300X_PEAKS) as (_, hw_name):
        train_efficiency_model(
            "gemm", PROF_DIR_MI / "gemm_mi300x_fp16_data.csv",
            str(MODEL_DIR_MI / f"gemm_{hw_name}_fp16_xgb.pth"),
            backend="xgboost", dtype="fp16",
        )
        train_efficiency_model(
            "attn", PROF_DIR_MI / "attn_mi300x_fp16_data.csv",
            str(MODEL_DIR_MI / f"attn_{hw_name}_fp16_xgb.pth"),
            backend="xgboost", dtype="fp16",
        )
        train_efficiency_model(
            "rmsnorm", PROF_DIR_MI / "rmsnorm_mi300x_fp16_data.csv",
            str(MODEL_DIR_MI / f"rmsnorm_{hw_name}_fp16_xgb.pth"),
            backend="xgboost", dtype="fp16",
        )
        set_efficiency_model_dir(str(MODEL_DIR_MI))

        r_trained_mi = syssim.simulate(model=QWEN3, hardware=MI300X,
                                       parallelism=PAR_MI, training=TR)
        trained_ms_mi = r_trained_mi.step_time_ms
finally:
    set_efficiency_model_dir("")

pd.DataFrame([
    {"estimator": "Roofline-only", "step_time_ms": round(roofline_ms_mi, 2), "delta_pct": "—"},
    {"estimator": "Trained predictor (synth)",
     "step_time_ms": round(trained_ms_mi, 2),
     "delta_pct": f"{100 * (trained_ms_mi - roofline_ms_mi) / roofline_ms_mi:+.1f}%"},
])

## §4. Precision FP8 (Dayou)

Same workflow as §3, but the dimension is precision rather than vendor.
H100's `peak_tflops_mm_fp8` (3958) gives ~2× throughput vs bf16.

In [ ]:
# §4a: FP8 roofline
TR_FP8 = syssim.TrainingConfig(micro_batch=1, global_batch=8, dtype="fp8")

print("=== H100 FP8 — Roofline-only ===")
r_roof_fp8 = syssim.simulate(model=QWEN3, hardware=HW, parallelism=PAR, training=TR_FP8)
print(f"step_time_ms={r_roof_fp8.step_time_ms:.2f}  mfu={r_roof_fp8.mfu:.3f}  "
      f"peak_mem_gb={r_roof_fp8.peak_memory_gb:.2f}")
roofline_ms_fp8 = r_roof_fp8.step_time_ms

In [ ]:
# §4b: Synthesize H100 FP8 profiling data (1 byte per FP8 element)
H100_FP8_PEAKS = {
    "peak_tflops_mm": 1979.0, "peak_tflops_math": 989.0,
    "peak_memory_bandwidth_gbps": 3350.0, "peak_tflops_mm_fp8": 3958.0,
    "peak_tflops_mm_fp4": None,
}

PROF_DIR_FP8 = Path(tempfile.mkdtemp(prefix="h100_fp8_profiling_"))
helpers.synthesize_gemm_csv(PROF_DIR_FP8 / "gemm_h100_fp8_data.csv",
                            peak_tflops=3958, peak_bw_GBps=3350, dtype_bytes=1)
helpers.synthesize_attn_csv(PROF_DIR_FP8 / "attn_h100_fp8_data.csv",
                            peak_tflops=3958, peak_bw_GBps=3350, dtype_bytes=1)
helpers.synthesize_rmsnorm_csv(PROF_DIR_FP8 / "rmsnorm_h100_fp8_data.csv",
                               peak_bw_GBps=3350, dtype_bytes=1)
print(f"Wrote FP8 profiling CSVs to {PROF_DIR_FP8}")

In [ ]:
# §4c: Train FP8 predictor → swap → re-simulate
import os
MODEL_DIR_FP8 = Path(tempfile.mkdtemp(prefix="h100_fp8_models_"))

try:
    with helpers.simulated_hardware("h100", H100_FP8_PEAKS) as (_, hw_name):
        train_efficiency_model(
            "gemm", PROF_DIR_FP8 / "gemm_h100_fp8_data.csv",
            str(MODEL_DIR_FP8 / f"gemm_{hw_name}_fp8_xgb.pth"),
            backend="xgboost", dtype="fp8",
        )
        train_efficiency_model(
            "attn", PROF_DIR_FP8 / "attn_h100_fp8_data.csv",
            str(MODEL_DIR_FP8 / f"attn_{hw_name}_fp8_xgb.pth"),
            backend="xgboost", dtype="fp8",
        )
        train_efficiency_model(
            "rmsnorm", PROF_DIR_FP8 / "rmsnorm_h100_fp8_data.csv",
            str(MODEL_DIR_FP8 / f"rmsnorm_{hw_name}_fp8_xgb.pth"),
            backend="xgboost", dtype="fp8",
        )
        set_efficiency_model_dir(str(MODEL_DIR_FP8))
        os.environ["SYSSIM_FORCE_DTYPE"] = "fp8"
        try:
            r_trained_fp8 = syssim.simulate(model=QWEN3, hardware=HW,
                                            parallelism=PAR, training=TR_FP8)
            trained_ms_fp8 = r_trained_fp8.step_time_ms
        finally:
            del os.environ["SYSSIM_FORCE_DTYPE"]
finally:
    set_efficiency_model_dir("")  # reset for §5

pd.DataFrame([
    {"estimator": "FP8 Roofline-only", "step_time_ms": round(roofline_ms_fp8, 2), "delta_pct": "—"},
    {"estimator": "FP8 Trained predictor (synth)",
     "step_time_ms": round(trained_ms_fp8, 2),
     "delta_pct": f"{100 * (trained_ms_fp8 - roofline_ms_fp8) / roofline_ms_fp8:+.1f}%"},
])

## §5. Cost model — modifying `estimate_runtime()` (Dayou)

SysSim's per-op estimator is a Protocol (`syssim.compute.estimator.Estimator`)
with one method: `estimate_op(...)`. The default is `RooflineEstimator`;
custom backends (e.g. PLENA at `syssim/external/plena/backend.py:282`)
implement the same protocol and slot in via `HardwareConfig.estimator`.

Below: a toy `ConstantEstimator` that returns 1ms per op.

In [ ]:
# The pluggable estimator protocol
import inspect
from syssim.compute.estimator import Estimator, RooflineEstimator
print(inspect.getsource(Estimator))

In [ ]:
# Our toy estimator (defined once in demo/helpers.py)
print(inspect.getsource(helpers.ConstantEstimator))

# Load the H100 config and attach the custom estimator
from syssim.training.spec import load_hardware_yaml
hw_cfg = load_hardware_yaml(HW)
hw_cfg.estimator = helpers.ConstantEstimator(constant_ms=1.0)

r_const = syssim.simulate(model=QWEN3, hardware=hw_cfg, parallelism=PAR, training=TR)
print(f"With ConstantEstimator(1ms): step_time_ms = {r_const.step_time_ms:.2f}")
print(f"(Compare to §1 Qwen3-8B bf16 roofline.)")

For a real custom estimator, see [`syssim/external/plena/backend.py`](https://github.com/AISysSim/SysSim/blob/master/syssim/external/plena/backend.py)
— PLENA maps PyTorch ops to cycle-level performance on a custom accelerator
using the same `Estimator` protocol.